# Vanilla Neural Networks: Medical Appointment No-Show Prediction

This notebook implements a custom neural network from scratch to predict medical appointment no-shows using the Kaggle medical appointment dataset.

## Dataset
The dataset contains information about medical appointments and whether patients showed up or not.

In [ ]:
%pip install ydata-profiling
import pandas as pd
import ydata_profiling as pp

## Data Loading and Initial Exploration

In [ ]:
df = pd.read_csv('KaggleV2-May-2016.csv')
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])
df['DayOfWeek'] = df['AppointmentDay'].dt.day_name()
print("Dataset shape:", df.shape)
print("First few rows:")
df.head()

## Data Preprocessing

### Feature Selection
Removing irrelevant columns and focusing on useful features.

In [ ]:
# Drop irrelevant columns
df.drop(columns=['PatientId', 'AppointmentID', 'ScheduledDay', 'AppointmentDay', 'Neighbourhood'], inplace=True)
print("Columns after dropping:", df.columns.tolist())
df.head()

### Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Label encode the target variable
le = LabelEncoder()
df['No-show'] = le.fit_transform(df['No-show'])  # 'No' -> 0, 'Yes' -> 1
print("Target distribution:")
print(df['No-show'].value_counts())

# One-hot encode categorical features
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'No-show']
print("Categorical columns to encode:", categorical_cols)

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
print("Shape after encoding:", df.shape)
df.head()

### Correlation Analysis

In [ ]:
# Calculate correlation with target
correlations = df.corr(numeric_only=True)['No-show'].sort_values(ascending=False)
print("Feature correlations with No-show:")
print(correlations)

### Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df['Age'] = scaler.fit_transform(df[['Age']])
print("Age column scaled.")
df.head()

## Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('No-show', axis=1)
y = df['No-show']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Training target distribution:", y_train.value_counts(normalize=True))
print("Test target distribution:", y_test.value_counts(normalize=True))

## Neural Network Implementation

### Helper Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

def get_class_weights(y):
    weights = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return {i: weight for i, weight in enumerate(weights)}

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def sigmoid_derivative(Z):
    s = sigmoid(Z)
    return s * (1 - s)

### Parameter Initialization

In [ ]:
def initialize_parameters(layer_dims):
    np.random.seed(3)
    parameters = {}
    L = len(layer_dims)
    for l in range(1, L):
        parameters['W' + str(l)] = np.random.randn(layer_dims[l-1], layer_dims[l]) * 0.1
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
    return parameters

### Forward Propagation

In [ ]:
def linear_forward(A_prev, W, b):
    Z = np.dot(W.T, A_prev) + b
    A = sigmoid(Z)
    cache = (A_prev, W, b, Z)
    return A, cache

def L_layer_forward(X, parameters):
    A = X
    caches = []
    L = len(parameters) // 2
    for l in range(1, L + 1):
        A_prev = A
        W = parameters['W' + str(l)]
        b = parameters['b' + str(l)]
        A, cache = linear_forward(A_prev, W, b)
        caches.append(cache)
    return A, caches

### Loss Function

In [ ]:
def compute_loss(y, y_hat):
    m = y.shape[1]
    loss = -np.sum(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8)) / m
    return loss

### Backward Propagation and Parameter Update

In [ ]:
def update_parameters(parameters, caches, y_true, y_hat, learning_rate, class_weight):
    L = len(caches)
    m = y_hat.shape[1]
    dA = -(class_weight) * (y_true / y_hat - (1 - y_true) / (1 - y_hat))
    
    for l in reversed(range(L)):
        A_prev, W, b, Z = caches[l]
        dZ = dA * sigmoid_derivative(Z)
        dW = np.dot(A_prev, dZ.T)
        db = np.sum(dZ, axis=1, keepdims=True)
        dA = np.dot(W, dZ)
        parameters['W' + str(l+1)] -= learning_rate * dW
        parameters['b' + str(l+1)] -= learning_rate * db

## Training Configuration

In [ ]:
# Network architecture
layer_dims = [X_train.shape[1], 3, 3, 1]  # Input -> Hidden1 -> Hidden2 -> Output
parameters = initialize_parameters(layer_dims)

# Class weights for imbalanced dataset
class_weights = get_class_weights(y_train)
class_weights[1] *= 4  # Amplify minority class weight
print("Class weights:", class_weights)

# Training hyperparameters
initial_lr = 0.01
final_lr = 0.00001
epochs = 400
batch_size = 200

# Initialize tracking variables
train_losses = []
test_losses = []

## Training Loop

In [ ]:
for i in range(epochs):
    # Linear learning rate decay
    learning_rate = initial_lr - ((initial_lr - final_lr) / epochs) * i
    
    # Forward pass on training data
    y_hat_train, _ = L_layer_forward(X_train.T, parameters)
    y_hat_test, _ = L_layer_forward(X_test.T, parameters)
    
    # Compute and store loss
    train_loss = compute_loss(y_train.values.reshape(1, -1), y_hat_train)
    test_loss = compute_loss(y_test.values.reshape(1, -1), y_hat_test)
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    # Mini-batch updates
    n_batches = X_train.shape[0] // batch_size
    for batch in range(n_batches):
        start = batch * batch_size
        end = start + batch_size
        x_batch = X_train.iloc[start:end].values.T
        y_batch = y_train.iloc[start:end].values.reshape(1, -1)
        y_hat_batch, caches = L_layer_forward(x_batch, parameters)
        weights = np.array([class_weights[y] for y in y_batch.flatten()])
        avg_weight = np.mean(weights)
        update_parameters(parameters, caches, y_batch, y_hat_batch, learning_rate=learning_rate, class_weight=avg_weight)
    
    # Print progress every 50 epochs
    if (i + 1) % 50 == 0:
        print(f"Epoch {i+1}/{epochs}, Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

## Model Evaluation

In [ ]:
# Final predictions
y_hat_train_final, _ = L_layer_forward(X_train.T, parameters)
y_hat_test_final, _ = L_layer_forward(X_test.T, parameters)

# Convert probabilities to binary predictions with threshold tuning
thresholds = np.arange(0.05, 1.05, 0.05)
probs = y_hat_test_final.flatten()

print("Threshold\tCount (< threshold)")
for th in thresholds:
    count = np.sum(probs > th)
    print(f"{th:.2f}\t\t{count}")

# Use 0.18 as threshold for better balance
y_pred_train = (y_hat_train_final.flatten() > 0.18).astype(int)
y_pred_test = (y_hat_test_final.flatten() > 0.18).astype(int)

print("\n=== Final Results ===")
print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print("F1 Score (Test):", f1_score(y_test, y_pred_test, zero_division=0))
print("Confusion Matrix (Test):\n", confusion_matrix(y_test, y_pred_test))

## Training Visualization

In [ ]:
# Plot training curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Train Loss", color='blue')
plt.plot(test_losses, label="Test Loss", color='red')
plt.xlabel("Epochs")
plt.ylabel("Binary Cross-Entropy Loss")
plt.title("Training and Test Loss Curves")
plt.legend()
plt.grid(True)
plt.show()